# Limpieza y preparación de datos


Este notebook realiza la limpieza y preparación de los datos para el entrenamiento del clasificador de facturas electrónicas, incluyendo la normalización de las descripciones, la conservación de la información relevante y la identificación de posibles incidencias.

In [1]:
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import unicodedata
from decimal import Decimal
from hashlib import sha256
from pathlib import Path

import pandas as pd
from IPython.display import display

In [3]:
candidates = [
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "ecuador-einvoice-classifier")
]

PROJECT_DIR = next(
    (
        path
        for path in candidates
        if (path / "ml/notebooks/01_data_extractor.ipynb").is_file()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Abre el notebook desde el proyecto o su carpeta de trabajo."
    )

ML_DIR = PROJECT_DIR / "ml"

In [4]:
EXTRACTED_DIR = ML_DIR / "data/extracted"
CLEANED_DIR = ML_DIR / "data/cleaned"
KEY = ["source_file_path", "item_position"]

raw_invoices = pd.read_parquet(EXTRACTED_DIR / "invoices.parquet")
raw_invoice_items = pd.read_parquet(EXTRACTED_DIR / "details.parquet")

print(
    f"Entrada: {len(raw_invoices):,} archivos y {len(raw_invoice_items):,} conceptos etiquetados."
)

Entrada: 7,225 archivos y 22,796 conceptos etiquetados.


## 1. Integridad de los datos


Se verifica que todas las facturas provengan del mismo paquete de datos y que el paquete no haya cambiado desde la extracción.


In [5]:
CATEGORIES = [
    "Alimentación",
    "Salud",
    "Educación, Arte y Cultura",
    "Vivienda",
    "Vestimenta",
    "Turismo Nacional",
    "Otros",
]

if raw_invoices.empty or raw_invoice_items.empty:
    raise ValueError("No hay facturas o conceptos para preparar.")

dataset_ids = raw_invoices["dataset_archive_sha256"].dropna().unique()

if len(dataset_ids) != 1 or raw_invoices["dataset_archive_sha256"].isna().any():
    raise ValueError("Las facturas no identifican un único .zip de origen.")

package_hash = sha256((ML_DIR / "data/invoices_dataset.zip").read_bytes()).hexdigest()

if package_hash != dataset_ids[0]:
    raise ValueError("El .zip cambió. Revisa primero la extracción y el EDA.")

In [6]:
if raw_invoices["source_file_id"].duplicated().any():
    raise ValueError("Hay identificadores de factura repetidos.")

if raw_invoice_items.duplicated(KEY).any():
    raise ValueError("Hay claves de concepto repetidas, revisa la extracción.")

if not raw_invoice_items["category"].isin(CATEGORIES).all():
    raise ValueError("Hay categorías ausentes o fuera del catálogo revisado.")

if not raw_invoice_items["source_file_id"].isin(raw_invoices["source_file_id"]).all():
    raise ValueError("Hay conceptos sin factura de origen.")

if raw_invoices["issue_date"].isna().any():
    raise ValueError("Hay fechas no interpretables, revisa la extracción.")

if raw_invoice_items["line_total"].isna().any():
    raise ValueError("Hay importes netos no interpretables, revisa la extracción.")

## 2. Consolidación de facturas duplicadas


Se verifica que no haya facturas duplicadas y se consolidan los datos de las facturas y de los conceptos asociados a ellas.


In [7]:
identity_variants = raw_invoices.groupby("anonymized_access_key")[
    "invoice_body_sha256"
].nunique()

if identity_variants.gt(1).any():
    raise ValueError("Hay una identificación con distintos contenidos, revisa eso.")

ordered_invoices = raw_invoices.sort_values("source_file_path").copy()
copy_keys = ["anonymized_access_key", "source_xml_sha256"]

ordered_invoices["retained_source_file_path"] = ordered_invoices.groupby(copy_keys)[
    "source_file_path"
].transform("first")

copy_mask = ordered_invoices.duplicated(copy_keys, keep="first")
copies = ordered_invoices.loc[copy_mask].copy()
invoices = (
    ordered_invoices.loc[~copy_mask].drop(columns="retained_source_file_path").copy()
)

In [8]:
items = raw_invoice_items.loc[
    raw_invoice_items["source_file_id"].isin(invoices["source_file_id"])
].copy()

items = items.sort_values(["source_file_id", "item_position"]).reset_index(drop=True)

excluded_records = raw_invoice_items.loc[
    raw_invoice_items["source_file_id"].isin(copies["source_file_id"])
].copy()

copy_representatives = copies.set_index("source_file_id")["retained_source_file_path"]
excluded_records["exclusion_reason"] = "exact_invoice_copy"
excluded_records["retained_source_file_path"] = excluded_records["source_file_id"].map(
    copy_representatives
)

print(
    f"Copias consolidadas: {len(copies):,} facturas y {len(excluded_records):,} conceptos."
)

Copias consolidadas: 21 facturas y 42 conceptos.


## 3. Preparación de las descripciones


Se normalizan los caracteres Unicode y los espacios en `normalized_description`, conservando la descripción original en `raw_description`, mientras que `description_comparison_text` permite identificar variantes de escritura y analizar textos repetidos sin eliminar conceptos ni sustituir las descripciones utilizadas durante el entrenamiento.


In [9]:
def clean_text(value: str) -> str:
    """Uniforma Unicode y espacios, conservando los demás elementos del texto."""
    normalized = unicodedata.normalize("NFKC", value)

    return " ".join(normalized.split())


def comparison_text(value: str) -> str:
    """Agrupa variantes de escritura para diagnóstico, sin eliminar compras."""
    normalized = clean_text(value)

    folded = unicodedata.normalize("NFKD", normalized.upper())
    folded = "".join(char for char in folded if not unicodedata.combining(char))

    key = " ".join(re.findall(r"[^\W_]+", folded))

    return key or "literal:" + normalized

In [10]:
items["normalized_description"] = items["raw_description"].map(clean_text)
items["description_comparison_text"] = items["raw_description"].map(comparison_text)

items["has_alphanumeric_description"] = items["normalized_description"].str.contains(
    r"[^\W_]"
)
items["has_anonymization_marker"] = items["raw_description"].str.contains(
    r"ANONIMIZAD[OA]", case=False, regex=True
)

## 4. Datos del contexto de los conceptos de la factura


El EDA identificó que el 20,9 % de las facturas contiene conceptos de distintas categorías, por lo que el contexto puede aportar información adicional sin asumir que todas las líneas pertenecen al mismo rubro. Para ello, `other_items_normalized_description_text` reúne las descripciones normalizadas de los demás conceptos en su orden original, excluyendo la línea que se desea clasificar y las categorías de las otras, mientras que las descripciones breves o anonimizadas se conservan.


In [11]:
def other_descriptions(group: pd.DataFrame) -> pd.Series:
    """Reúne las otras descripciones normalizadas de la factura, respetando su orden."""
    texts = group["normalized_description"].tolist()
    contexts = []

    for position in range(len(texts)):
        other_texts = texts[:position] + texts[position + 1 :]
        contexts.append(" | ".join(other_texts))

    return pd.Series(contexts, index=group.index, dtype="string")

In [ ]:
context_parts = []

for _, invoice_group in items.groupby("source_file_id", sort=False):
    context_parts.append(other_descriptions(invoice_group))

items["other_items_normalized_description_text"] = pd.concat(context_parts).reindex(
    items.index
)

text_fields = [
    "normalized_description",
    "product_code",
    "auxiliary_product_code",
    "additional_item_attributes",
    "other_items_normalized_description_text",
]

available_text = items[text_fields].fillna("").agg(" ".join, axis=1)

items["has_alphanumeric_input_text"] = available_text.str.contains(r"[^\W_]")

print(
    f"Descripciones sin caracteres alfanuméricos: "
    f"{int((~items['has_alphanumeric_description']).sum()):,}"
)

print(
    f"Conceptos sin texto alfanumérico disponible: "
    f"{int((~items['has_alphanumeric_input_text']).sum()):,}"
)

Descripciones sin caracteres alfanuméricos: 3
Conceptos sin texto alfanumérico disponible: 0


## 5. Moneda e importes


El EDA identificó 57 facturas y 131 conceptos cuya moneda era ambigua, por lo que se excluyen de los cálculos económicos en USD, conservando el resto de sus campos. Solo se reconocen las denominaciones explícitas de USD, mientras que las demás mantienen `recognized_currency_code` sin valor y `has_unrecognized_currency`.


In [13]:
USD_NAMES = {
    "DOLAR",
    "DOLARES",
    "USD",
    "DOLLAR",
    "DOLLARS",
    "DOLAR AMERICANO",
    "DOLAR US",
    "US DOLLAR",
    "DOLAR USA",
    "US DOLARES",
}

currency_keys = invoices["raw_currency"].map(comparison_text)
recognized_usd = currency_keys.isin(USD_NAMES)

invoices["recognized_currency_code"] = pd.Series(
    pd.NA, index=invoices.index, dtype="string"
)
invoices.loc[recognized_usd, "recognized_currency_code"] = "USD"
invoices["has_unrecognized_currency"] = ~recognized_usd

## 6. Conservación y verificación de los valores numéricos


Los valores numéricos se conservan con su precisión decimal, sin eliminar ceros, modificar importes ni excluir valores extremos. Para comprobar su consistencia, se compara el importe neto declarado con el resultado de multiplicar la cantidad por el precio unitario y restar el descuento.


In [14]:
def item_net_amount_difference(row: pd.Series) -> Decimal | None:
    """Compara el importe calculado con el declarado, sin corregirlo."""
    fields = ["quantity", "unit_price", "discount", "line_total"]

    if row[fields].isna().any():
        return None

    calculated_total = row["quantity"] * row["unit_price"] - row["discount"]

    return calculated_total - row["line_total"]


def exceeds_one_cent(value: Decimal | None) -> bool:
    """Indica si una diferencia calculable merece revisión."""
    if value is None:
        return False

    return abs(value) > Decimal("0.01")

In [15]:
items["item_net_amount_difference"] = items.apply(item_net_amount_difference, axis=1)
items["has_item_net_amount_difference"] = items["item_net_amount_difference"].map(
    exceeds_one_cent
)
items["has_zero_line_amount"] = items["line_total"].eq(Decimal(0))

numeric_fields = ["quantity", "unit_price", "discount", "line_total"]
missing_numbers = items[numeric_fields].isna().any(axis=1)
negative_numbers = items[numeric_fields].lt(0).any(axis=1)

items["has_missing_or_negative_numeric_values"] = missing_numbers | negative_numbers

## 7. Construcción y validación de la tabla de conceptos


La tabla de conceptos se genera conservando los registros originales, excluyendo las copias exactas e incorporando la fecha y moneda de cada factura. Posteriormente, se verifica que cada concepto esté contabilizado una sola vez entre los registros conservados y excluidos, y que sus etiquetas, importes y demás campos mantengan sus valores originales.


In [16]:
invoice_context_columns = [
    "source_file_id",
    "issue_date",
    "raw_currency",
    "recognized_currency_code",
    "has_unrecognized_currency",
]

model_dataset = items.merge(
    invoices[invoice_context_columns],
    on="source_file_id",
    validate="many_to_one",
)

concept_counts = model_dataset.groupby("source_file_id").size()

invoices["prepared_item_count"] = (
    invoices["source_file_id"].map(concept_counts).fillna(0)
)
invoices["prepared_item_count"] = invoices["prepared_item_count"].astype("int64")

In [17]:
accounted_keys = pd.concat(
    [model_dataset[KEY], excluded_records[KEY]],
    ignore_index=True,
)

if accounted_keys.duplicated(KEY).any():
    raise RuntimeError(
        "Se detectó un concepto duplicado entre los registros conservados y excluidos."
    )

original_columns = raw_invoice_items.columns.tolist()
accounted_records = pd.concat(
    [model_dataset[original_columns], excluded_records[original_columns]],
    ignore_index=True,
)

original_records = raw_invoice_items.set_index(KEY).sort_index()
accounted_records = accounted_records.set_index(KEY).sort_index()

if not original_records.equals(accounted_records):
    raise RuntimeError(
        "Los datos preparados presentan diferencias respecto a los campos originales."
    )

if not model_dataset["source_file_id"].isin(invoices["source_file_id"]).all():
    raise RuntimeError("Hay conceptos preparados sin factura de origen.")

## 8. Revisión de la distribución de categorías y almacenamiento de resultados


Se revisa la distribución de conceptos por categoría para identificar los cambios tras la depuración.


In [18]:
class_balance = pd.DataFrame(
    {
        "Entrada": raw_invoice_items["category"].value_counts(),
        "Conservados": model_dataset["category"].value_counts(),
        "Excluidos por copia": excluded_records["category"].value_counts(),
    }
)

class_balance = class_balance.reindex(CATEGORIES).fillna(0).astype(int)

display(class_balance)

,Entrada,Conservados,Excluidos por copia
category,,,
Alimentación,9942,9939,3
Salud,1515,1514,1
"Educación, Arte y Cultura",945,945,0
Vivienda,1930,1915,15
Vestimenta,358,358,0
Turismo Nacional,168,167,1
Otros,7938,7916,22


In [19]:
quality_summary = pd.Series(
    {
        "Descripciones sin letras ni dígitos": int(
            (~model_dataset["has_alphanumeric_description"]).sum()
        ),
        "Descripciones con marcas de anonimización": int(
            model_dataset["has_anonymization_marker"].sum()
        ),
        "Conceptos con moneda no identificada": int(
            model_dataset["has_unrecognized_currency"].sum()
        ),
        "Conceptos con importe cero": int(model_dataset["has_zero_line_amount"].sum()),
        "Diferencias aritméticas superiores a un centavo": int(
            model_dataset["has_item_net_amount_difference"].sum()
        ),
        "Conceptos con números ausentes o negativos": int(
            model_dataset["has_missing_or_negative_numeric_values"].sum()
        ),
    },
    name="Conceptos conservados",
)

display(quality_summary.to_frame())

,Conceptos conservados
Descripciones sin letras ni dígitos,3
Descripciones con marcas de anonimización,99
Conceptos con moneda no identificada,131
Conceptos con importe cero,587
Diferencias aritméticas superiores a un centavo,102
Conceptos con números ausentes o negativos,0


In [20]:
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

prepared_tables = {
    "invoices": invoices,
    "model_dataset": model_dataset,
    "excluded_records": excluded_records,
}

for name, frame in prepared_tables.items():
    output_path = CLEANED_DIR / f"{name}.parquet"

    frame.to_parquet(output_path, index=False)

    restored = pd.read_parquet(output_path)

    if len(restored) != len(frame) or list(restored.columns) != list(frame.columns):
        raise RuntimeError(f"La lectura de vuelta no coincide: {name}.")

    print(f"Guardado {output_path.relative_to(PROJECT_DIR)} - {len(frame):,} filas.")

Guardado ml/data/cleaned/invoices.parquet - 7,204 filas.


Guardado ml/data/cleaned/model_dataset.parquet - 22,754 filas.
Guardado ml/data/cleaned/excluded_records.parquet - 42 filas.


In [21]:
example_columns = [
    "raw_description",
    "normalized_description",
    "category",
    "quantity",
    "line_total",
    "has_anonymization_marker",
]

display(model_dataset[example_columns].head(8))


,raw_description,normalized_description,category,quantity,line_total,has_anonymization_marker
0,chifle,chifle,Alimentación,1.00000000,0.430000000000,False
1,Encebollado,Encebollado,Alimentación,1.00000000,1.960000000000,False
2,COM.PAGO SERVICIOS,COM.PAGO SERVICIOS,Otros,1.00000000,0.270000000000,False
3,VEGA ANNI/B18/31-ENE-25/11:50,VEGA ANNI/B18/31-ENE-25/11:50,Turismo Nacional,1.00000000,9.800000000000,False
4,PRESERVATIVOS FIVE SKIN CAJA X 5,PRESERVATIVOS FIVE SKIN CAJA X 5,Salud,1.00000000,2.630000000000,False
5,AFEITADO REPUESTO MAQ GILLETTE MACH 3 X 2 SENS...,AFEITADO REPUESTO MAQ GILLETTE MACH 3 X 2 SENS...,Otros,1.00000000,6.770000000000,False
6,LECHE DESLACTOSADA LA LECHERA TETRABRIK 946 ML...,LECHE DESLACTOSADA LA LECHERA TETRABRIK 946 ML...,Alimentación,1.00000000,1.750000000000,False
7,SILICON 250 ML LANCER,SILICON 250 ML LANCER,"Educación, Arte y Cultura",1.00000000,1.960000000000,False


In [22]:
print(f"Facturas procesadas: {len(invoices):,}")
print(f"Conceptos conservados: {len(model_dataset):,}")
print(f"Conceptos excluidos por duplicidad: {len(excluded_records):,}")


Facturas procesadas: 7,204
Conceptos conservados: 22,754
Conceptos excluidos por duplicidad: 42


## Archivos generados y descripción de sus columnas


### 1. `invoices.parquet`

Contiene una fila por factura conservada.

| Columna                     | Descripción                                                                             |
| --------------------------- | --------------------------------------------------------------------------------------- |
| `source_file_id`            | Identificador generado a partir de la ruta del archivo XML dentro del ZIP.              |
| `source_file_path`          | Ruta del archivo XML dentro del ZIP de origen.                                          |
| `anonymized_access_key`     | Clave de acceso anonimizada presente en el XML.                                         |
| `dataset_archive_sha256`    | Huella SHA-256 del archivo ZIP completo.                                                |
| `source_xml_sha256`         | Huella SHA-256 del XML completo, utilizada para detectar copias exactas.                |
| `invoice_body_sha256`       | Huella SHA-256 del contenido canónico de la factura, sin el envoltorio de autorización. |
| `invoice_version`           | Versión del XML de la factura.                                                          |
| `issue_date_raw`            | Fecha de emisión en su formato original.                                                |
| `issue_date`                | Fecha de emisión interpretada como fecha.                                               |
| `raw_currency`              | Denominación monetaria original.                                                        |
| `recognized_currency_code`  | Código USD cuando se reconoce la moneda.                                                |
| `has_unrecognized_currency` | Indica si la moneda no pudo identificarse como USD.                                     |
| `original_item_count`       | Cantidad de conceptos del comprobante original.                                         |
| `net_total_raw`             | Importe total sin impuestos en su formato original.                                     |
| `net_total`                 | Importe total sin impuestos interpretado como Decimal.                                  |
| `total_discount_raw`        | Descuento total en su formato original.                                                 |
| `total_discount`            | Descuento total interpretado como Decimal.                                              |
| `invoice_total_raw`         | Importe final de la factura en su formato original.                                     |
| `invoice_total`             | Importe final interpretado como Decimal.                                                |
| `tip_raw`                   | Propina en su formato original, cuando está disponible.                                 |
| `tip`                       | Propina interpretada como Decimal, cuando está disponible.                              |
| `tax_amount`                | Suma de los impuestos registrados en la cabecera de la factura.                         |
| `prepared_item_count`       | Cantidad de conceptos conservados en el conjunto preparado.                             |

### 2. `model_dataset.parquet`

Contiene una fila por concepto conservado.

| Columna                                   | Descripción                                                                                                                  |
| ----------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------- |
| `source_file_id`                          | Identificador del archivo XML al que pertenece el concepto.                                                                  |
| `source_file_path`                        | Ruta del archivo XML de origen dentro del ZIP.                                                                               |
| `anonymized_access_key`                   | Clave de acceso anonimizada de la factura.                                                                                   |
| `item_position`                           | Posición original del concepto dentro del XML, comenzando en 1.                                                              |
| `raw_description`                         | Descripción original del concepto sin modificaciones.                                                                        |
| `product_code`                            | Código principal del producto o servicio.                                                                                    |
| `auxiliary_product_code`                  | Código auxiliar del producto o servicio, cuando existe.                                                                      |
| `additional_item_attributes`              | Atributos adicionales del concepto con sus nombres y valores.                                                                |
| `quantity_raw`                            | Cantidad declarada en su formato original.                                                                                   |
| `quantity`                                | Cantidad interpretada como Decimal.                                                                                          |
| `unit_price_raw`                          | Precio unitario en su formato original.                                                                                      |
| `unit_price`                              | Precio unitario interpretado como Decimal.                                                                                   |
| `discount_raw`                            | Descuento del concepto en su formato original.                                                                               |
| `discount`                                | Descuento del concepto interpretado como Decimal.                                                                            |
| `line_total_raw`                          | Importe neto del concepto en su formato original.                                                                            |
| `line_total`                              | Importe neto declarado después de descuentos y antes de impuestos, interpretado como Decimal.                                |
| `tax_amount`                              | Suma de los impuestos asociados al concepto.                                                                                 |
| `category`                                | Categoría revisada y asignada manualmente al concepto.                                                                       |
| `issue_date`                              | Fecha de emisión de la factura de origen.                                                                                    |
| `raw_currency`                            | Denominación monetaria original de la factura.                                                                               |
| `recognized_currency_code`                | Código USD cuando se reconoce la moneda.                                                                                     |
| `has_unrecognized_currency`               | Indica si la moneda no pudo identificarse como USD.                                                                          |
| `normalized_description`                  | Descripción con Unicode y espacios normalizados, conservando los demás caracteres.                                           |
| `description_comparison_text`             | Representación normalizada para comparar variantes de escritura y detectar repeticiones.                                     |
| `has_alphanumeric_description`            | Indica si la descripción contiene letras o números.                                                                          |
| `has_anonymization_marker`                | Indica si la descripción contiene el marcador explícito de anonimización.                                                    |
| `other_items_normalized_description_text` | Descripciones normalizadas de los demás conceptos de la factura, conservando su orden original y sin incluir sus categorías. |
| `has_alphanumeric_input_text`             | Indica si existe texto alfanumérico en la descripción, códigos, atributos o contexto.                                        |
| `item_net_amount_difference`              | Diferencia entre el importe calculado mediante cantidad, precio y descuento y el importe neto declarado.                     |
| `has_item_net_amount_difference`          | Indica si la diferencia aritmética absoluta supera 0,01.                                                                     |
| `has_zero_line_amount`                    | Indica si el importe neto declarado del concepto es cero.                                                                    |
| `has_missing_or_negative_numeric_values`  | Indica la presencia de cantidades, precios, descuentos o importes netos ausentes o negativos.                                |

### 3. `excluded_records.parquet`

Contiene los conceptos de las copias exactas excluidas.

| Columna                      | Descripción                                                                          |
| ---------------------------- | ------------------------------------------------------------------------------------ |
| `source_file_id`             | Identificador del archivo XML excluido.                                              |
| `source_file_path`           | Ruta del archivo XML excluido dentro del ZIP.                                        |
| `anonymized_access_key`      | Clave de acceso anonimizada de la factura.                                           |
| `item_position`              | Posición original del concepto dentro de la factura.                                 |
| `raw_description`            | Descripción original del concepto.                                                   |
| `product_code`               | Código principal del producto o servicio.                                            |
| `auxiliary_product_code`     | Código auxiliar del producto o servicio.                                             |
| `additional_item_attributes` | Atributos adicionales del concepto.                                                  |
| `quantity_raw`               | Cantidad en su formato original.                                                     |
| `quantity`                   | Cantidad interpretada como Decimal.                                                  |
| `unit_price_raw`             | Precio unitario en su formato original.                                              |
| `unit_price`                 | Precio unitario interpretado como Decimal.                                           |
| `discount_raw`               | Descuento del concepto en su formato original.                                       |
| `discount`                   | Descuento interpretado como Decimal.                                                 |
| `line_total_raw`             | Importe neto del concepto en su formato original.                                    |
| `line_total`                 | Importe neto interpretado como Decimal.                                              |
| `tax_amount`                 | Suma de los impuestos asociados al concepto.                                         |
| `category`                   | Categoría revisada del concepto, conservada sin modificaciones.                      |
| `exclusion_reason`           | Motivo de exclusión, identificado como `exact_invoice_copy` para las copias exactas. |
| `retained_source_file_path`  | Ruta del XML conservado como representante de la factura duplicada.                  |
